In [ ]:
# ============================================================
# 07_shap_vs_dice.ipynb
# Feature-attribution (SHAP) vs counterfactual (DiCE) comparison:
# why counterfactual recourse, not attribution, satisfies the
# interpretability/recourse control for underwriting.
#
# SHAP identifies the features that most drive predicted risk
# (attribution, no target value); DiCE identifies the actionable
# changes that would lower it (recourse, with target values). We
# quantify their overlap (Jaccard of SHAP Top-3 vs DiCE changed
# features) and contrast target-value availability.
#
# Input : data/processed/final_pipeline_{sex}.pkl,
#         data/processed/highrisk_nondiabetic_{sex}.csv,
#         data/processed/counterfactuals_{sex}.csv
# Output: results/tables/shap_vs_dice.csv,
#         results/tables/shap_importance_{sex}.csv,
#         results/figures/shap_importance_{sex}.(png|pdf)
#
# Figures grayscale, dpi 600, PNG + PDF, no captions.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
for p in (FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)

GRAY = ["#000000", "#595959", "#8C8C8C", "#BFBFBF", "#D9D9D9"]
sns.set_theme(style="whitegrid", context="paper")
mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600, "savefig.bbox": "tight",
    "font.family": "serif", "font.size": 10, "axes.edgecolor": "black",
    "axes.linewidth": 0.8, "axes.grid": True, "grid.color": "0.88", "grid.linewidth": 0.6,
})
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

cat_col = ["HE_obe","BO1_1","BO1_2","BO1_3","BD1_11","BD2_1","BS3_1","BE3_71",
           "BE3_75","BE3_81","BE3_91","pa_aerobic","L_BR_FQ","BP1","mh_stress",
           "incm","ho_incm","edu","BH1"]
num_col = ["HE_BMI","HE_wc","HE_wt","N_SUGAR","N_CHO","N_EN","age"]
feature_col = cat_col + num_col

def orig_feature(name):
    # map 'num__HE_BMI' / 'cat__HE_obe_1.0' back to original feature
    base = name.split("__", 1)[-1]
    for c in sorted(feature_col, key=len, reverse=True):
        if base == c or base.startswith(c + "_"):
            return c
    return base

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else 0.0

rows = []
for sex in ["male", "female"]:
    bundle = joblib.load(PROC / f"final_pipeline_{sex}.pkl")
    pipe = bundle["pipe"]
    prep, clf = pipe.named_steps["prep"], pipe.named_steps["clf"]
    hr = pd.read_csv(PROC / f"highrisk_nondiabetic_{sex}.csv")[feature_col]
    Xp = prep.transform(hr)
    feat_names = prep.get_feature_names_out()

    # TreeSHAP on the XGBoost step
    explainer = shap.TreeExplainer(clf)
    sv = explainer.shap_values(Xp)
    if isinstance(sv, list):
        sv = sv[1]
    mean_abs = np.abs(sv).mean(axis=0)

    # aggregate |SHAP| to original features
    agg = {}
    for name, val in zip(feat_names, mean_abs):
        o = orig_feature(name)
        agg[o] = agg.get(o, 0.0) + float(val)
    imp = pd.Series(agg).sort_values(ascending=False)
    imp.rename("mean_abs_shap").to_csv(TAB / f"shap_importance_{sex}.csv")
    shap_top3 = list(imp.index[:3])

    # DiCE changed features per case
    cf = pd.read_csv(PROC / f"counterfactuals_{sex}.csv")
    jac = []
    for _, r in cf.iterrows():
        dice_feats = str(r.get("changed_features", "") or "").split("|")
        dice_feats = [f for f in dice_feats if f]
        jac.append(jaccard(shap_top3, dice_feats))
    actionable_in_top3 = sum(1 for f in shap_top3 if f != "age")
    rows.append(dict(
        Sex=sex, SHAP_Top3="|".join(shap_top3),
        SHAP_provides_target_value="No",
        DiCE_provides_target_value="Yes",
        Mean_Jaccard_SHAP_vs_DiCE=round(float(np.mean(jac)) if jac else 0.0, 3),
        SHAP_top3_actionable=actionable_in_top3))
    print(f"{sex:7s}: SHAP Top-3 = {shap_top3} | mean Jaccard vs DiCE = "
          f"{np.mean(jac) if jac else 0:.3f}")

    # ---- Figure: SHAP global importance (Top-10, grayscale) ----
    top = imp.head(10)[::-1]
    fig, ax = plt.subplots(figsize=(5.2, 4.0))
    colors = [GRAY[0] if f == "age" else GRAY[3] for f in top.index]
    ax.barh(range(len(top)), top.values, color=colors, edgecolor="black", height=0.7)
    ax.set_yticks(range(len(top))); ax.set_yticklabels(top.index)
    ax.set_xlabel("Mean |SHAP value|")
    save_fig(fig, f"shap_importance_{sex}")

cmp = pd.DataFrame(rows)
cmp.to_csv(TAB / "shap_vs_dice.csv", index=False)
print("\nSHAP vs DiCE comparison:\n", cmp.to_string(index=False))
print("Figures written: shap_importance_{male,female} (png+pdf).")
